In [0]:
# Escrevendo a Tabela Delta Silver.

from pyspark.sql.functions import col, lower, initcap, month, year, when, count, regexp_replace
from pyspark.sql.types import DoubleType, IntegerType

In [0]:
# Acessando a Tabela Delta Bronze. 

df_bronze = spark.table("projeto_profissional_bigdata.datatran_2018.bronze_datatran18")

In [0]:
# Visualizando o Schema

df_bronze.printSchema()

In [0]:
# Selecionando as colunas

df_silver = df_bronze.select(
    "id"
    ,"data_inversa"
    ,"uf"
    ,"br"
    ,"km"
    ,"municipio"
    ,"causa_acidente"
    ,"tipo_acidente"
    ,"classificacao_acidente"
    ,"pessoas"
    ,"feridos"
    ,"mortos"
)

df_silver.show(5)
df_silver.printSchema()

In [0]:
# Removendo linhas com valores nulos.

df_silver = df_silver.dropna(
    subset=["id","data_inversa","uf","br","km","municipio","causa_acidente","tipo_acidente","classificacao_acidente","pessoas","feridos","mortos"]
)
df_silver.show(5)

In [0]:
# Renomeando coluna data inversa para data acidente.

df_silver = df_silver.withColumnRenamed("data_inversa", "data_acidente")
df_silver.show(5)

In [0]:
# Padronizndo nome de colunas.

df_silver = df_silver.withColumn(
    "municipio",
    initcap(col("municipio")))
df_silver = df_silver.withColumn(
    "causa_acidente",
    lower(col("causa_acidente")))
df_silver = df_silver.withColumn( 
    "tipo_acidente",
    lower(col("tipo_acidente")))

df_silver.show(5)

In [0]:

 # criando a coluna mês acidente

df_silver = df_silver.withColumn(
"mes_acidente",
month(col("data_acidente")))

df_silver.show(5)


In [0]:
# criando a coluna ano acidente

df_silver = df_silver.withColumn(
"ano_acidente",
year(col("data_acidente")))

df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn(
    "nome_mes",
    when(col("mes_acidente") == 1, "Janeiro")
    .when(col("mes_acidente") == 2, "Fevereiro")
    .when(col("mes_acidente") == 3, "Março")
    .when(col("mes_acidente") == 4, "Abril")
    .when(col("mes_acidente") == 5, "Maio")
    .when(col("mes_acidente") == 6, "Junho")
    .when(col("mes_acidente") == 7, "Julho")
    .when(col("mes_acidente") == 8, "Agosto")
    .when(col("mes_acidente") == 9, "Setembro")
    .when(col("mes_acidente") == 10, "Outubro")
    .when(col("mes_acidente") == 11, "Novembro")
    .when(col("mes_acidente") == 12, "Dezembro")
    .otherwise("Dezembro")
)
df_silver.show(5)


In [0]:

# Contando Linhas da Coluna BR NA  

df_silver.filter(col("br") == "NA").count()

In [0]:
# Contando Linhas da Coluna BR = NA por None   

df_silver = df_silver.withColumn(
    "br",
    when(col("br") == "NA", None)
    .otherwise(col("br")))

In [0]:
df_silver = df_silver.dropna(subset=["br"])
# Contando Linhas da Coluna KM NA  

In [0]:
df_silver = df_silver.withColumn(
    "br",
    col("br").cast(IntegerType())
)

df_silver.printSchema()
df_silver.show(5)

In [0]:
df_silver = df_silver.withColumn(
    "km",
    regexp_replace(col("km"), ",", "").cast(DoubleType())
)
df_silver.printSchema()
df_silver.show(10)


In [0]:
df_silver.write\
    .format("delta")\
.mode("overwrite")\
.saveAsTable("projeto_profissional_bigdata.datatran_2018.silver_datatran18")

In [0]:
df = spark.table("projeto_profissional_bigdata.datatran_2018.silver_datatran18")
df.show()

In [0]:
%sql
SELECT * FROM projeto_profissional_bigdata.datatran_2018.silver_datatran18